# Day 19 - A*, and what happens when a search stops being sequential

Dijkstra orders its frontier by `g(v)`, the cost already paid. A* orders it by

$$f(v) = g(v) + h(v)$$

where `h(v)` is a *guess* at what is left. That is the whole change - one term
added to the priority - and it is the first time in this series that an
algorithm is allowed to be wrong on purpose.

Two properties decide whether the guess is safe:

| property | condition | what it buys |
|---|---|---|
| admissible | `h(v)` never exceeds the true remaining cost | the answer is still optimal |
| consistent | `h(u) <= w(u,v) + h(v)` for every edge | a vertex never has to be re-expanded |

The second half of the notebook asks the same question in parallel: Dijkstra's
strict one-vertex-at-a-time order is exactly what makes it hard to parallelise,
and delta-stepping is the dial between "too strict" and "too much work".

In [1]:
import heapq
from collections import deque

# '#' is a wall.  A digit is the cost of STEPPING ONTO that cell.
# '5' is mud: passable, but five times the price of open ground.
GRID = [
    "111111111",
    "111111111",
    "115555511",
    "111111111",
    "111111111",
    "111111111",
]
START = (2, 0)
GOAL = (2, 8)
MIN_COST = 1                      # the cheapest any single step can ever be

def parse(grid):
    rows, cols = len(grid), len(grid[0])
    cost = [[None] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            ch = grid[r][c]
            cost[r][c] = None if ch == '#' else int(ch)
    return rows, cols, cost

ROWS, COLS, COST = parse(GRID)
STEPS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def neighbours(cell):
    r, c = cell
    for dr, dc in STEPS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and COST[nr][nc] is not None:
            yield (nr, nc), COST[nr][nc]


def show(expanded=(), path=(), title=''):
    exp, pth = set(expanded), set(path)
    print(title)
    for r in range(ROWS):
        row = []
        for c in range(COLS):
            cell = (r, c)
            if COST[r][c] is None:
                row.append('##')
            elif cell == START:
                row.append(' S')
            elif cell == GOAL:
                row.append(' G')
            elif cell in pth:
                row.append(' *')
            elif cell in exp:
                row.append(' o')
            elif COST[r][c] > 1:
                row.append(' ~')
            else:
                row.append(' .')
        print('   ' + ''.join(row))

show(title='   S start, G goal, ## wall, ~ mud (cost 5), . open (cost 1)')

   S start, G goal, ## wall, ~ mud (cost 5), . open (cost 1)
    . . . . . . . . .
    . . . . . . . . .
    S . ~ ~ ~ ~ ~ . G
    . . . . . . . . .
    . . . . . . . . .
    . . . . . . . . .


## Dijkstra is A* with h = 0

The loop below is a single function. Set `h = 0` and it is Dijkstra; hand it a
heuristic and it is A*; multiply the heuristic and it is weighted A*; ignore
`g` altogether and it is greedy best-first. Nothing else changes.

In [2]:
def manhattan(cell, goal=GOAL):
    """|dr| + |dc|, scaled by the cheapest possible step.

    Admissible: any route to the goal needs at least this many steps, and no
    step costs less than MIN_COST, so the true remaining cost is never smaller.
    """
    return (abs(cell[0] - goal[0]) + abs(cell[1] - goal[1])) * MIN_COST


def best_first(start, goal, h=None, weight=1.0, greedy=False):
    """Dijkstra, A*, weighted A* and greedy best-first - all the same loop.

    The only thing that changes is the priority:

        greedy      f = h(v)                 ignore what we have paid
        otherwise   f = g(v) + weight*h(v)   weight 1 and h=0 -> Dijkstra
    """
    if h is None:
        h = lambda cell: 0
    g = {start: 0}
    parent = {start: None}
    key = (lambda cell, gv: h(cell)) if greedy else (lambda cell, gv: gv + weight * h(cell))
    pq = [(key(start, 0), start)]
    closed = set()
    expanded = []                       # every vertex we pop and scan
    while pq:
        _, u = heapq.heappop(pq)
        if u in closed:
            continue
        closed.add(u)
        expanded.append(u)
        if u == goal:
            break
        for v, w in neighbours(u):
            ng = g[u] + w
            if v not in g or ng < g[v]:
                g[v] = ng
                parent[v] = u
                heapq.heappush(pq, (key(v, ng), v))
    if goal not in g:
        return None, 0, expanded
    path, cur = [], goal
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    return path[::-1], g[goal], expanded


def path_cost(path):
    return sum(COST[r][c] for r, c in path[1:])

dij_path, dij_cost, dij_exp = best_first(START, GOAL)
a_path,  a_cost,  a_exp  = best_first(START, GOAL, manhattan)

show(dij_exp, dij_path, 'Dijkstra: expanded %d cells, cost %d' % (len(dij_exp), dij_cost))
print()
show(a_exp, a_path, 'A* (Manhattan): expanded %d cells, cost %d' % (len(a_exp), a_cost))
print()
print('same cost, %.0f%% of the work' % (100.0 * len(a_exp) / len(dij_exp)))

Dijkstra: expanded 49 cells, cost 10
    o o o o o o o o o
    * * * * * * * * *
    S o o o o ~ ~ o G
    o o o o o o o o o
    o o o o o o o o .
    o o o o o o o . .

A* (Manhattan): expanded 13 cells, cost 10
    . . . . . . . . .
    o * * * * * * * *
    S * ~ ~ ~ ~ ~ o G
    . . . . . . . . .
    . . . . . . . . .
    . . . . . . . . .

same cost, 27% of the work


## Is the heuristic allowed?

Manhattan distance times the cheapest possible step never overestimates, because
any route needs at least that many steps and no step is cheaper than `MIN_COST`.
It is also consistent, which is the stronger property - and the one that
licenses the `closed` set. Without consistency a vertex can be popped while its
`g` is not yet final, and a correct implementation has to be willing to reopen
it.

In [3]:
def is_admissible(h, true_dist):
    """h(v) <= the real remaining cost, for every reachable v."""
    return all(h(v) <= d for v, d in true_dist.items())


def is_consistent(h):
    """h(u) <= w(u,v) + h(v) for every edge - the triangle inequality.

    Consistency is the stronger property, and it is the one that licenses the
    `closed` set: pop a vertex once and its g is final.
    """
    for r in range(ROWS):
        for c in range(COLS):
            if COST[r][c] is None:
                continue
            u = (r, c)
            for v, w in neighbours(u):
                if h(u) > w + h(v):
                    return False, (u, v, h(u), w + h(v))
    return True, None


def true_distances(goal):
    """Dijkstra from the goal on the reversed graph - the exact h we could
    never afford to compute in advance."""
    dist = {goal: 0}
    pq = [(0, goal)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, _ in neighbours(u):
            # a step is charged for the cell it enters, so walking the graph
            # backwards we pay for the cell we are leaving
            nd = d + COST[u[0]][u[1]]
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

td = true_distances(GOAL)
print('admissible:', is_admissible(manhattan, td))
print('consistent:', is_consistent(manhattan)[0])
print('h(start) =', manhattan(START), ' true remaining cost =', td[START])

admissible: True
consistent: True
h(start) = 8  true remaining cost = 10


## Two ways to be fast and wrong

Inflating the heuristic makes the search greedier. Weighted A* keeps a promise -
the path costs at most `w` times the optimum - while greedy best-first, which
throws `g` away completely, promises nothing at all. Neither raises, neither
returns an error code, and both hand back a perfectly plausible path.

In [4]:
w_path, w_cost, w_exp = best_first(START, GOAL, manhattan, weight=5.0)
gr_path, gr_cost, gr_exp = best_first(START, GOAL, manhattan, greedy=True)

show(w_exp, w_path, 'weighted A* (h x 5): cost %d, expanded %d' % (w_cost, len(w_exp)))
print()
print('weight   cost   expanded   optimal?')
for weight in (1, 2, 3, 4, 5, 8):
    p, c, e = best_first(START, GOAL, manhattan, weight=float(weight))
    print('%-8d %-6d %-10d %s' % (weight, c, len(e),
          'yes' if c == a_cost else 'NO (%.1fx optimal, bound was %dx)' % (c / a_cost, weight)))
print('%-8s %-6d %-10d %s' % ('greedy', gr_cost, len(gr_exp),
      'NO (%.1fx optimal, no bound at all)' % (gr_cost / a_cost)))

weighted A* (h x 5): cost 28, expanded 9
    . . . . . . . . .
    . . . . . . . . .
    S * * * * * * * G
    . . . . . . . . .
    . . . . . . . . .
    . . . . . . . . .

weight   cost   expanded   optimal?
1        10     13         yes
2        10     11         yes
3        10     12         yes
4        10     15         yes
5        28     9          NO (2.8x optimal, bound was 5x)
8        28     9          NO (2.8x optimal, bound was 8x)
greedy   28     9          NO (2.8x optimal, no bound at all)


## Delta-stepping: the dial between Dijkstra and Bellman-Ford

Dijkstra settles exactly one vertex per step, and that strict order is precisely
what a parallel machine cannot exploit. Bellman-Ford happily relaxes everything
at once but does far too much work. Delta-stepping buckets tentative distances
by width `delta` and settles a whole bucket as one batch.

In [5]:
def grid_adj():
    """The grid, rewritten as a plain adjacency dict - so the parallel routines
    below can run on any weighted graph, not just this map."""
    adj = {}
    for r in range(ROWS):
        for c in range(COLS):
            if COST[r][c] is not None:
                adj[(r, c)] = list(neighbours((r, c)))
    return adj

# A small weighted graph with a deliberate mix of light and heavy edges.
# The grid is nearly unweighted, so it hides what delta-stepping is for.
WEIGHTED_EDGES = [
    (0, 1, 7), (0, 3, 1), (0, 5, 3), (0, 7, 7),
    (0, 13, 1), (1, 2, 1), (1, 4, 7), (1, 10, 1),
    (2, 6, 1), (2, 11, 1), (3, 2, 1), (4, 12, 1),
    (5, 8, 7), (5, 13, 1), (7, 8, 3), (8, 9, 7),
    (10, 12, 1), (11, 10, 9), (12, 7, 9), (12, 10, 2),
    (12, 11, 3), (13, 2, 2), (13, 11, 7),
]

def weighted_adj(edges=WEIGHTED_EDGES):
    adj = {}
    for a, b, w in edges:
        adj.setdefault(a, []).append((b, w))
        adj.setdefault(b, []).append((a, w))
    return adj


def delta_stepping(adj, start, delta):
    """Dijkstra relaxes one vertex at a time; Bellman-Ford relaxes everything.

    Delta-stepping puts tentative distances into buckets of width delta and
    settles a whole bucket at once - every vertex in it may be relaxed in
    parallel.  Light edges (w <= delta) can move a vertex within the current
    bucket, so the bucket is re-scanned until it stops changing; heavy edges
    are deferred to the end of the phase, when distances can no longer shrink.

    delta -> 0        one vertex per bucket   = Dijkstra
    delta -> infinity one bucket for the lot  = Bellman-Ford

    Returns the distances, the phases (each phase is a batch that could have
    been relaxed in parallel) and a count of edge relaxations - the price paid
    for the wider batches.
    """
    dist = {start: 0}
    buckets = {0: {start}}
    phases = []                      # (bucket index, vertices settled together)
    relaxations = 0
    b = 0
    while buckets:
        if b not in buckets:
            b = min(buckets)
        heavy = []
        while buckets.get(b):
            frontier = buckets.pop(b)
            phases.append((b, sorted(frontier)))
            for u in frontier:           # <- one parallel batch
                for v, w in adj[u]:
                    relaxations += 1
                    nd = dist[u] + w
                    if v not in dist or nd < dist[v]:
                        if w <= delta:
                            if v in dist:
                                old = dist[v] // delta
                                if old in buckets:
                                    buckets[old].discard(v)
                                    if not buckets[old]:
                                        del buckets[old]
                            dist[v] = nd
                            buckets.setdefault(nd // delta, set()).add(v)
                        else:
                            heavy.append((u, v, w))
        for u, v, w in heavy:            # heavy edges, once per phase
            relaxations += 1
            nd = dist[u] + w
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                buckets.setdefault(nd // delta, set()).add(v)
        buckets = {k: st for k, st in buckets.items() if st}
        if buckets:
            b = min(buckets)
    return dist, phases, relaxations

wadj = weighted_adj()
print('delta   phases   largest batch   edge relaxations')
ref = None
for delta in (1, 2, 4, 9, 99):
    dist, phases, relax = delta_stepping(wadj, 0, delta)
    ref = ref or dist
    assert dist == ref, 'delta must not change the answer'
    print('%-7d %-8d %-15d %d' % (delta, len(phases), max(len(v) for _, v in phases), relax))

delta   phases   largest batch   edge relaxations
1       10       3               54
2       9        3               54
4       8        3               61
9       8        5               66
99      5        8               82


## BFS as a frontier, not a queue

The parallel version of BFS is not "a queue with locks". It is the observation
that every vertex in the current level is independent, so a level can be
expanded all at once. The *work* is unchanged at O(V+E); what improves is the
*depth* - the number of levels - and that is what a machine with enough cores
actually waits for.

In [6]:
def parallel_bfs_levels(adj, start):
    """BFS written the way a parallel implementation has to write it.

    Not "a queue", but "a frontier": every vertex in the current level is
    independent, so the whole level can be expanded at once.  The sequential
    cost is unchanged - O(V+E) work - but the *depth* is the number of levels,
    which is what a machine with enough cores actually waits for.
    """
    seen = {start}
    frontier = [start]
    levels = []
    while frontier:
        levels.append(sorted(frontier))
        nxt = []
        for u in frontier:               # <- this loop is the parallel part
            for v, _ in adj[u]:
                if v not in seen:
                    seen.add(v)
                    nxt.append(v)
        frontier = nxt
    return levels

gadj = grid_adj()
levels = parallel_bfs_levels(gadj, START)
print('%d levels, widths %s' % (len(levels), [len(l) for l in levels]))
print('work  = O(V+E)')
print('depth = %d levels' % len(levels))
print('widest frontier = %d independent vertices' % max(len(l) for l in levels))

12 levels, widths [1, 3, 5, 6, 6, 6, 6, 6, 6, 5, 3, 1]
work  = O(V+E)
depth = 12 levels
widest frontier = 6 independent vertices


## LeetCode 1091 - Shortest Path in Binary Matrix

Movement is 8-directional, so the admissible guess is the Chebyshev distance
`max(|dr|, |dc|)`, not Manhattan - Manhattan counts a diagonal step twice, which
overestimates, which is exactly the way to lose optimality. On an empty board
Chebyshev is *exact*, and A* walks straight to the goal without expanding
anything else.

In [7]:
def shortest_path_binary_matrix(grid):
    """8-directional, unit cost, 0 = open.  A* with the Chebyshev distance.

    max(|dr|, |dc|) is exactly the number of diagonal-ish steps needed on an
    empty board, so it is admissible and consistent - and on an empty board it
    is exact, which means A* walks straight to the goal.
    """
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    if n == 1:
        return 1
    goal = (n - 1, n - 1)
    h = lambda c: max(abs(c[0] - goal[0]), abs(c[1] - goal[1]))
    g = {(0, 0): 1}
    pq = [(1 + h((0, 0)), (0, 0))]
    closed = set()
    expanded = 0
    while pq:
        _, u = heapq.heappop(pq)
        if u in closed:
            continue
        closed.add(u)
        expanded += 1
        if u == goal:
            return g[u]
        r, c = u
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                v = (r + dr, c + dc)
                if 0 <= v[0] < n and 0 <= v[1] < n and grid[v[0]][v[1]] == 0:
                    ng = g[u] + 1
                    if v not in g or ng < g[v]:
                        g[v] = ng
                        heapq.heappush(pq, (ng + h(v), v))
    return -1


def bfs_binary_matrix(grid):
    """The usual BFS answer, for cross-checking A*."""
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    q = deque([((0, 0), 1)])
    seen = {(0, 0)}
    while q:
        (r, c), d = q.popleft()
        if (r, c) == (n - 1, n - 1):
            return d
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                v = (r + dr, c + dc)
                if (0 <= v[0] < n and 0 <= v[1] < n and grid[v[0]][v[1]] == 0
                        and v not in seen):
                    seen.add(v)
                    q.append((v, d + 1))
    return -1

boards = [
    [[0, 1], [1, 0]],
    [[0, 0, 0], [1, 1, 0], [1, 1, 0]],
    [[1, 0, 0], [1, 1, 0], [1, 1, 0]],
]
for b in boards:
    print('%-34s A* %-3s BFS %s' % (b, shortest_path_binary_matrix(b), bfs_binary_matrix(b)))

[[0, 1], [1, 0]]                   A* 2   BFS 2
[[0, 0, 0], [1, 1, 0], [1, 1, 0]]  A* 4   BFS 4
[[1, 0, 0], [1, 1, 0], [1, 1, 0]]  A* -1  BFS -1


## Tests

In [8]:
assert dij_cost == a_cost, 'an admissible heuristic must not change the cost'
assert len(a_exp) < len(dij_exp), 'A* should look at less of the map'
assert is_admissible(manhattan, td) and is_consistent(manhattan)[0]
assert w_cost > a_cost and gr_cost > a_cost, 'the inflated guesses should be worse'
assert w_cost <= 5 * a_cost, 'weighted A* must honour its own bound'
assert path_cost(a_path) == a_cost
exact = delta_stepping(gadj, START, 1)[0]
for delta in (2, 5, 99):
    assert delta_stepping(gadj, START, delta)[0] == exact, 'delta changed the answer'
assert exact[GOAL] == a_cost
assert sum(len(l) for l in levels) == len(gadj)
for b in boards:
    assert shortest_path_binary_matrix(b) == bfs_binary_matrix(b)
print('all assertions passed')

all assertions passed
